In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
from optbinning import ContinuousOptimalBinning

from sklearn.metrics import (
    mean_squared_error
)

In [ ]:
df = pd.read_csv('src/data/training/processed/final_training_data.csv')

In [ ]:
target_cols = df.filter(regex='^(dn__|ul__|mv__|madrid__|ep__|cb__|ob__|ab__)').columns

missing_or_zero = df[target_cols].isna() | (df[target_cols] == 0)

missing_fraction = missing_or_zero.mean(axis=1)

df = df[missing_fraction <= 0.5].copy()

In [ ]:
df = df.drop(columns=['user_id','reference_date'])  # Drop non-feature columns

In [ ]:
reg_feats = ['t__ccf', 'ob__open_limit_ref', 'ul__account_age_months']
clf_feats = ['ob__avg_util_ref', 'ob__open_limit_ref', 'ob__util_slope_3m', 'ob__max_util_0_1m', 'ab__days_down_6m', 'ab__max_one_day_drop_6m', 'ab__std_abs_change_6m', 'ab__mean_abs_change_6m', 'ab__std_util_0_1m', 'ab__near_zero_days_6m']

In [ ]:
df = df[reg_feats]

In [ ]:

# Columns to filter (all except 't__ccf')
cols_to_filter = df.columns.difference(['t__ccf'])

# Compute 0.9 quantile for these columns
quantiles = df[cols_to_filter].quantile(0.99)

# Filter rows where all selected columns <= 0.9 quantile
df_filtered = df[(df[cols_to_filter] <= quantiles).all(axis=1)]

# Keep the filtered columns + target
df_filtered = df_filtered[cols_to_filter.tolist() + ['t__ccf']]

print(f"Original shape: {df.shape}")
print(f"Filtered shape: {df_filtered.shape}")

In [ ]:
df_filtered

In [ ]:
y = df_filtered[['t__ccf']].clip(lower=0, upper=1)
y = y.squeeze()          # converts DataFrame (n,1) -> Series (n,)
y = y.values.ravel()
X = df_filtered.drop(columns=['t__ccf'])

In [ ]:
df_filtered

In [ ]:
def supervised_bin_and_dummy(X, y, max_bins=5):
    # force y to be 1D
    y = pd.Series(y).squeeze().values.ravel()

    binned_arrays = []
    binners = {}

    for col in X.columns:
        optb = ContinuousOptimalBinning(name=col, dtype="numerical", max_n_bins=max_bins)
        optb.fit(X[col], y)

        # Transform to bin index
        binned = optb.transform(X[col], metric="indices").astype(int).ravel()
        binned_arrays.append(binned)
        binners[col] = optb

    X_binned = pd.DataFrame(np.column_stack(binned_arrays), columns=X.columns)

    X_dummies = pd.get_dummies(X_binned, columns=X.columns, prefix=X.columns)

    return X_dummies, binners

In [ ]:
X_dummies, binners = supervised_bin_and_dummy(X, y, max_bins=3)

In [ ]:
X_dummies = X_dummies.astype(int)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_dummies, y, test_size=0.2, random_state=42)

In [ ]:
binners

In [ ]:
X_train

In [ ]:
#X_train_sm = sm.add_constant(X_train)
model = sm.OLS(y_train, X_train).fit()
print(model.summary())

In [ ]:
X_train